# AAR Legibility PVG — frozen Claude prover

Same pipeline as `Experiments_v2.ipynb`, with one change: **Claude writes the findings
instead of a fine-tuned Qwen**. Claude Opus 5 (`anthropic/claude-opus-5` via OpenRouter)
gets the same helpful and sneaky prompts, its weights never change, and only the verifier
(Qwen2.5-1.5B + LoRA) trains round over round. This is the mode that matches the real AAR
(LIMITATIONS.md #5): can a small trained verifier keep up with a strong, fixed sneaky prover
and still accept the honest write-ups?

**Why Opus 5.** In a 24-sample check (26 Sep) its honest write-ups were all accurate and
carefully caveated, and its sneaky ones all misrepresented the data, mostly by citing a
number next to the real one (0.80 for 0.79) and by overclaiming. A larger model would not
write a better one-paragraph finding from eight numbers, and Fable 5.1 costs twice as much.
To try another model, change `CLAUDE_MODEL` in section 1.

**Before anything: Runtime → Change runtime type → T4 GPU.** Run order **1 → 2 → 3 → 4**, then 5.

**Cost** (measured): about $0.012 per completion, 64 completions per round
(8 records × 4 samples × 2 roles), so about $0.80 per round and about $25 for
10 rounds × 3 seeds, plus about $1 for the preview and smoke test. Completions are cached
on disk under the checkpoint folder, so re-running a round after a disconnect is free.
Section 1 prints the key's remaining budget.

**Time:** 3 to 6 minutes per round on a T4 (almost all of it verifier training, which grows
as the replay window fills), so roughly 2 to 3 hours for the full run.


## 1. Setup

In [ ]:
import os
from pathlib import Path

REPO_URL  = "https://github.com/varchanaiyer/Automated_R_leg_NoAPI.git"   # after merge: https://github.com/gunasti2002/Automated_R_leg_NoAPI.git
BRANCH    = "fix/verifier-scoring-and-gates"                               # after merge: main
CLONE_DIR = Path("/content/Automated_R_leg_NoAPI")

%cd /content
!rm -rf {CLONE_DIR}
!git clone --branch {BRANCH} {REPO_URL} {CLONE_DIR}

matches = sorted(CLONE_DIR.rglob("config.py"))
assert matches, f"No config.py found under {CLONE_DIR}"
PROJECT_ROOT = matches[0].parent
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)


In [ ]:
# Colab ships torchao 0.10, which peft refuses (needs >= 0.16). Nothing here uses it: remove it.
!pip uninstall -y -q torchao
# torch + matplotlib ship with Colab; do not reinstall torch (CUDA wheel mismatch risk).
!pip install -q jinja2 "transformers>=4.50" peft accelerate
import importlib; importlib.invalidate_caches()

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


In [ ]:
# Frozen Claude prover: key + config overrides. Subprocesses (run_experiment -> run_one_round) inherit these.
import os, json
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY") or "PASTE-YOUR-OPENROUTER-KEY-HERE"
CLAUDE_MODEL = "anthropic/claude-opus-5"

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ["PVG_CONFIG_OVERRIDES"] = json.dumps({
    "use_finetunable_prover": False,       # Claude writes; only the verifier trains
    "prover_api_model": CLAUDE_MODEL,
    "prover_api_reasoning_effort": "low",
    # Verifier labels come from the ROLE here (helpful -> SOUND, sneaky -> UNSOUND).
    # In the 24-sample check every Claude helpful write-up was honest and every sneaky one
    # misrepresented the data; the rule check agreed on 23 of 24 but its one miss was a
    # sneaky write-up, which rule-check labelling would have dropped from training.
    "verifier_labels_from_rule_check": False,
})

# Key status and remaining credit (read-only calls).
import urllib.request
def _get(path):
    req = urllib.request.Request("https://openrouter.ai/api/v1" + path,
                                 headers={"Authorization": "Bearer " + OPENROUTER_API_KEY, "User-Agent": "Mozilla/5.0"})
    return json.load(urllib.request.urlopen(req, timeout=30))
k = _get("/auth/key")["data"]; c = _get("/credits")["data"]
print(f"key limit ${k.get('limit')}  used ${k.get('usage')}  remaining ${k.get('limit_remaining')}")
print(f"org credits left: {c['total_credits'] - c['total_usage']:.2f}")


## 2. Data

Same builder and split as v2 (120 pairs, 20% held out by record, seed 42).

In [ ]:
!python data/build_findings_dataset.py --target-pairs 120 --seed 42 \
    --out data/findings_pvg_dataset.jsonl --heldout-out data/findings_pvg_heldout.jsonl \
    --manifest-out data/dataset_manifest.json
!cat data/dataset_manifest.json


## 3. Preview Claude's write-ups, then the gate + baselines

Two paid calls (one per role, about $0.03) on the first training record so you can read
what Claude produces before spending on a run. The rule check verdict is printed under each.

In [ ]:
import sys, json
sys.path.insert(0, ".")
from config import PVGConfig
from training import pvg_loop
from training.api_prover import APIProver

cfg = PVGConfig()
assert not cfg.use_finetunable_prover, "PVG_CONFIG_OVERRIDES not picked up; re-run the setup cell"
prover = APIProver(cfg, "training/api_cache_preview")
rows = pvg_loop.load_dataset(cfg.output_dataset_path)
rec = next(r for r in rows if r["label"] == "honest")["experiment"]
print("record:", json.dumps(rec["metrics"]), "\n")
for role, tmpl in (("helpful", pvg_loop._HELPFUL_PROMPT_PATH), ("sneaky", pvg_loop._SNEAKY_PROMPT_PATH)):
    prompt = pvg_loop.render_prover_prompt(tmpl, rec, None, cfg)
    text = prover.complete(prompt, n=1, temperature=cfg.prover_temperature, cache_key=f"preview-{role}")[0]
    ok, issues = pvg_loop.rule_check(rec, text)
    print(f"--- {role.upper()} ({len(text.split())} words) ---\n{text}\n  rule check: {'passes' if ok else 'FAILS'} {issues}\n")
print("usage:", prover.usage)


In [ ]:
# Verifier spot check on the untrained verifier (expected to fail the 20-item bar; the warmed-up one must pass).
spot_rows = pvg_loop.load_dataset(cfg.spot_check_set_path)
report = pvg_loop.run_spot_check(cfg, spot_set_rows=spot_rows, enforce_set=False)


In [ ]:
# Rule-check and zero-shot-Qwen baselines on the held-out split (+ the 20-item set). Same as v2; skip if already run.
!python eval/baselines.py --out eval/baselines_heldout.json


In [ ]:
# Drop the notebook's verifier so the training subprocesses get the full card.
import gc, sys, torch
if "training.pvg_loop" in sys.modules:
    sys.modules["training.pvg_loop"]._VERIFIER_CACHE.clear()
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1024**3:.2f} / {total/1024**3:.2f} GiB")
assert free / total > 0.9, "Something is still holding GPU memory — restart the runtime."


## 4. Runs

Same runner and safety rails as v2: the spot-check gate runs at the start of every round,
and a seed stops itself if the held-out accept rate goes above 0.9 or the honest-minus-sneaky
reward gap drops to zero. Exit 2 = gate refused, exit 3 = a seed aborted itself.
Run the smoke test first; if it exits 2, stop and report the gate block.


In [ ]:
SEEDS          = [42, 43, 44]
TOTAL_ROUNDS   = 10
CHECKPOINT_DIR = "training/checkpoint_claude"


In [ ]:
# Smoke test — 1 round in one process, no checkpointing (about $0.80; its cache is training/api_cache).
!python training/pvg_loop.py --rounds 1 --seed 42
!cat training/pvg_round_history.jsonl


In [ ]:
# Full run. Safe to interrupt; re-run this cell to continue where it stopped (cached completions are free).
seeds = " ".join(str(s) for s in SEEDS)
!python run_experiment.py --seeds {seeds} --rounds {TOTAL_ROUNDS} --checkpoint-dir {CHECKPOINT_DIR}


## 5. Results

`role_fidelity_helpful` / `role_fidelity_sneaky` are diagnostics here: the share of Claude's
helpful write-ups that pass the rule check and of sneaky ones it catches. They do not
feed the labels in this mode.

In [ ]:
import subprocess, sys
from pathlib import Path

png = Path(CHECKPOINT_DIR) / "pvg_curves.png"
subprocess.run([sys.executable, "eval/plot_rounds.py", "--checkpoint-dir", CHECKPOINT_DIR, "--out", str(png)])
if png.exists():
    from IPython.display import Image, display
    display(Image(filename=str(png)))
else:
    print(f"No plot yet: no completed rounds under {CHECKPOINT_DIR} (gate refused, a seed aborted at round 1, or the run has not started).")


In [ ]:
import json, pandas as pd
from pathlib import Path
rows = []
for f in sorted(Path(CHECKPOINT_DIR).glob("seed_*/round_history.jsonl")):
    rows += [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
df = pd.DataFrame(rows)
if df.empty:
    print("No completed rounds yet.")
cols = ["seed", "round_number", "helpful_prover_accuracy", "verifier_robustness", "unparseable_share",
        "heldout_accept_rate", "heldout_accuracy", "heldout_auroc", "reward_gap",
        "role_fidelity_helpful", "role_fidelity_sneaky", "verifier_logit_bias", "aborted", "elapsed_s"]
df[[c for c in cols if c in df.columns]]


In [ ]:
# API spend per seed (cumulative), and a look at what Claude wrote in the latest round.
import json
from pathlib import Path
for f in sorted(Path(CHECKPOINT_DIR).glob("seed_*/api_usage.json")):
    u = json.loads(f.read_text())
    print(f"{f.parent.name}: ${u['cost_usd']:.2f}  calls={u['calls']}  cache_hits={u['cache_hits']}  refusals={u['refusals']}  errors={u['errors']}")
samples = sorted(Path(CHECKPOINT_DIR).glob("seed_*/samples/round_*.jsonl"))
if samples:
    last = [json.loads(l) for l in samples[-1].read_text().splitlines() if l.strip()]
    for s in [x for x in last if x["label"] == "helpful"][:2] + [x for x in last if x["label"] == "sneaky"][:2]:
        print(f"\n[{s['label']}] P(sound)={s['verifier_p_sound']:.2f} rule={'pass' if s['rule_check_passes'] else 'FAIL'}\n  {s['summary']}")


### Persisting results

Colab's disk is wiped when the runtime ends. Copy `training/checkpoint_claude/` to Drive,
or commit the `round_history.jsonl`, `api_usage.json`, `samples/` and the plot and push them.
Do not commit or share this notebook with the key filled in.
